## Importes y Conexión

In [ ]:
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time
from datetime import datetime

# Conectar a MongoDB
client = MongoClient('mongodb://localhost:27017/')

try:
    client.admin.command('ping')
    print("✓ Conexión exitosa a MongoDB")
except Exception as e:
    print(f"✗ Error en la conexión: {e}")

## Configuración de Base de Datos y Selectores

In [ ]:
# SELECCIONA LA BASE DE DATOS
BASE_DATOS_SELECCIONADA = "practica"  # ← CAMBIA AQUÍ: "practica" o "ej_proyecto"

db = client[BASE_DATOS_SELECCIONADA]
coleccion = db['G9_Agrotech_scraping']

print(f"Base de Datos: {db.name}")
print(f"Colección: {coleccion.name}")
print(f"Total de documentos actuales: {coleccion.count_documents({})}")

# CONFIGURACIÓN DEL SCRAPER
URL_OBJETIVO = "ESCRIBIR_AQUI_LA_URL"
SELECTOR_CONTENEDOR = "CLASE_DEL_BLOQUE_PRINCIPAL"
SELECTOR_DATO_A = "CLASE_DEL_NOMBRE_O_TITULO"
SELECTOR_DATO_B = "CLASE_DEL_VALOR_O_PRECIO"
GRUPO = "G9_Agrotech"

## Función para iniciar navegador

In [ ]:
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

print("✓ Función iniciar_navegador() lista")

## Ejecución del Scraper

In [ ]:
driver = iniciar_navegador()
contador_insertados = 0

try:
    print(f"Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5)
    
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f"Se encontraron {len(elementos)} registros potenciales.")
    
    for item in elementos:
        try:
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            
            valor_limpio = "".join(filter(str.isdigit, columna_b))
            
            documento = {
                "variable_nombre": columna_a,
                "variable_valor": valor_limpio,
                "status": 1.0,
                "fecha_captura": datetime.now().isoformat(),
                "grupo": GRUPO
            }
            
            coleccion.insert_one(documento)
            contador_insertados += 1
            print(f"  ✓ Registrado: {columna_a} = {valor_limpio}")
            
        except Exception as e:
            print(f"  ⚠ Error procesando elemento: {e}")
            continue
    
    print(f"\n{'='*50}")
    print(f"PROCESO COMPLETADO")
    print(f"{'='*50}")
    print(f"Total insertado: {contador_insertados}")
    print(f"Total en colección: {coleccion.count_documents({})}")

finally:
    driver.quit()
    print("Navegador cerrado.")

## Verificación de Datos

In [ ]:
print(f"\n{'='*50}")
print(f"VERIFICACIÓN DE DATOS")
print(f"{'='*50}")

# Ver últimos 5 documentos
print("\nÚltimos 5 documentos insertados:")
for doc in coleccion.find().sort("_id", -1).limit(5):
    print(f"  • {doc['variable_nombre']}: {doc['variable_valor']} ({doc['fecha_captura']})")

print(f"\nTotal de documentos en la colección: {coleccion.count_documents({})}")